# Exploratory Data Analysis — New Zealand University Funding

This notebook provides a univariate EDA of New Zealand tertiary funding, covering:

1. **TEC SAC rates** — government subsidy per EFTS (equivalent to AUS Commonwealth Contribution), Level 2 (undergraduate degree), excl. GST, 2016–2025.  
2. **Universities NZ domestic tuition fees** — programme-level min/max fees per institution, 2016, 2017, 2022–2024.

**Data sources:**
- `data/raw/NZ Funding/NZ_TEC_SAC_rates_all_years.csv` — TEC SAC government subsidy per EFTS by cost category
- `data/raw/NZ Funding/NZ_UniversitiesNZ_domestic_fees_all_years.csv` — per-university programme fees
- `data/raw/NZ Funding/NZ_UniversitiesNZ_domestic_fees_summary.csv` — annual aggregates

**Role in the JRGS project:** NZ funding context complements the enrolment data. Unlike AUS (where JRG explicitly repriced cost bands in 2021), NZ funding follows a separate TEC review cycle. Understanding NZ funding trends confirms there is no analogous funding shock in the control period.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')

# Resolve data directory
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
raw_nz_dir = None
for root in candidate_roots:
    candidate = root / 'data' / 'raw' / 'NZ Funding'
    if candidate.exists():
        raw_nz_dir = candidate
        break

if raw_nz_dir is None:
    raise FileNotFoundError(f'NZ Funding directory not found from: {Path.cwd()}')

# Load TEC SAC rates
sac = pd.read_csv(raw_nz_dir / 'NZ_TEC_SAC_rates_all_years.csv')

# Load Universities NZ fees
fees = pd.read_csv(raw_nz_dir / 'NZ_UniversitiesNZ_domestic_fees_all_years.csv')
fees_summary = pd.read_csv(raw_nz_dir / 'NZ_UniversitiesNZ_domestic_fees_summary.csv')

print('=== TEC SAC Rates ===')
print(f'Rows: {sac.shape[0]} | Years: {sorted(sac["year"].unique())} | Categories: {sorted(sac["category_code"].unique())}')
print('\n=== Universities NZ Fees ===')
print(f'Rows: {fees.shape[0]} | Years: {sorted(fees["year"].unique())} | Universities: {fees["university"].nunique()}')
print(sac.head())
print(fees.head())

## 1. Inspect Data Structure

Review structure of both the TEC SAC rates file and the Universities NZ fees file.

In [ ]:
print('=== TEC SAC Rates — dtypes ===')
print(sac.dtypes.to_frame('dtype'))
print(f'\nUnique categories: {sorted(sac["category_code"].unique())}')
print(f'Year range: {sac["year"].min()}–{sac["year"].max()}')
print('\nSample category descriptions:')
print(sac.groupby('category_code')['description'].first().to_string())

print('\n=== Universities NZ Fees — dtypes ===')
print(fees.dtypes.to_frame('dtype'))
print(f'\nUniversities: {sorted(fees["university"].unique())}')
print(f'Year range: {fees["year"].min()}–{fees["year"].max()}')
print('\nFees data info:')
fees.info()

## 2. Clean Missing Values and Duplicates

In [ ]:
# SAC rates — check for missing
sac_missing = sac.isna().sum()
print('SAC rates — missing values:')
print(sac_missing[sac_missing > 0] if sac_missing.any() else 'None')
print(f'SAC rate duplicates: {sac.duplicated().sum()}')

# Fees — check for missing
fees_missing = fees.isna().sum()
print('\nFees — missing values:')
print(fees_missing[fees_missing > 0] if fees_missing.any() else 'None')
print(f'Fees duplicates: {fees.duplicated().sum()}')

# Note which years are missing from fees (2018-2021 gap)
all_years = list(range(sac['year'].min(), fees['year'].max() + 1))
fee_years = sorted(fees['year'].unique())
missing_fee_years = [y for y in all_years if y not in fee_years]
print(f'\nYears with TEC SAC data: {sorted(sac["year"].unique())}')
print(f'Years with fee data: {fee_years}')
print(f'Fee data gaps: {missing_fee_years} (PDFs not publicly available)')

sac_clean  = sac.drop_duplicates().copy()
fees_clean = fees.drop_duplicates().copy()

## 3. Summary Statistics — TEC SAC Rates

Descriptive statistics for Level-2 SAC rates by category and year.

In [ ]:
print('=== SAC Rate summary statistics ===')
print(sac_clean['sac_rate_level2_excl_gst'].describe())

# Pivot: category × year
sac_pivot = (sac_clean.groupby(['category_code', 'year'])['sac_rate_level2_excl_gst']
             .first().unstack('year'))
print('\n=== SAC Rate pivot (Level-2, excl. GST) ===')
print(sac_pivot.to_string())

# First vs last year change
yr_min = sac_clean['year'].min()
yr_max = sac_clean['year'].max()
if yr_min in sac_pivot.columns and yr_max in sac_pivot.columns:
    sac_change = sac_pivot[[yr_min, yr_max]].copy()
    sac_change['Absolute Change'] = sac_change[yr_max] - sac_change[yr_min]
    sac_change['Percent Change']  = (sac_change['Absolute Change'] / sac_change[yr_min] * 100).round(2)
    print(f'\n=== SAC Rate change {yr_min}→{yr_max} ===')
    print(sac_change.sort_values('Percent Change', ascending=False).to_string())

## 4. Visualize TEC SAC Rate Trends

Plot government subsidy per EFTS (Level-2) by cost category over time.

In [ ]:
# Key categories with NZSCED field mapping
KEY_CATS = ['A', 'H', 'I', 'J', 'N', 'V', 'B', 'C']
key_sac = sac_clean[sac_clean['category_code'].isin(KEY_CATS)].copy()

# One-label per category for legend
cat_labels = key_sac.groupby('category_code')['description'].first().str[:40]
key_sac['cat_label'] = key_sac['category_code'].map(cat_labels)

# Line plot — key categories
plt.figure(figsize=(13, 6))
sns.lineplot(data=key_sac, x='year', y='sac_rate_level2_excl_gst',
             hue='cat_label', marker='o')
plt.title('NZ TEC SAC Rates (Level 2, excl. GST) — Key Cost Categories')
plt.ylabel('NZD per EFTS')
plt.xlabel('Year')
plt.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# All categories — heatmap
sac_pivot_all = (sac_clean.groupby(['category_code', 'year'])['sac_rate_level2_excl_gst']
                 .first().unstack('year'))
plt.figure(figsize=(14, 6))
sns.heatmap(sac_pivot_all, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('NZ TEC SAC Rates (Level 2, excl. GST) — All Categories')
plt.ylabel('Category Code')
plt.tight_layout()
plt.show()

# Indexed: base 2016 = 100
base_yr = 2016
if base_yr in sac_pivot.columns:
    sac_idx = sac_pivot.divide(sac_pivot[base_yr], axis=0) * 100
    sac_idx_long = sac_idx.reset_index().melt(id_vars='category_code', var_name='year', value_name='Index')
    sac_idx_long = sac_idx_long[sac_idx_long['category_code'].isin(KEY_CATS)].dropna()

    plt.figure(figsize=(13, 6))
    sns.lineplot(data=sac_idx_long, x='year', y='Index', hue='category_code', marker='o')
    plt.axhline(100, color='grey', ls=':', lw=1)
    plt.title(f'NZ TEC SAC Rates Indexed ({base_yr}=100) — Key Categories')
    plt.ylabel('Index')
    plt.tight_layout()
    plt.show()

## 5. Visualize Universities NZ Domestic Tuition Fee Trends

Plot average domestic tuition fees over time.

In [ ]:
print('=== Annual fee summary ===')
print(fees_summary.to_string(index=False))

# Line plot of avg min/max fee over time
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(fees_summary['year'], fees_summary['avg_of_min_fees'], marker='o', label='Avg min fee')
ax.plot(fees_summary['year'], fees_summary['avg_of_max_fees'], marker='s', ls='--', label='Avg max fee')
ax.fill_between(fees_summary['year'], fees_summary['avg_of_min_fees'],
                fees_summary['avg_of_max_fees'], alpha=0.15)
ax.set_xlabel('Year')
ax.set_ylabel('NZD')
ax.set_title('NZ Domestic Undergraduate Tuition Fees — Annual Average (Universities NZ)')
ax.legend()
plt.tight_layout()
plt.show()

# Fee distribution by university
plt.figure(figsize=(12, 6))
sns.boxplot(data=fees_clean, x='min_fee_nzd', y='university', whis=1.5, color='lightcoral')
sns.stripplot(data=fees_clean, x='min_fee_nzd', y='university', color='black', alpha=0.4, size=3)
plt.title('NZ Domestic Fee Distribution by University (Min Fee)')
plt.xlabel('Min Fee (NZD)')
plt.tight_layout()
plt.show()

# Fee by year (boxplot)
plt.figure(figsize=(10, 5))
sns.boxplot(data=fees_clean, x='year', y='min_fee_nzd', whis=1.5, palette='Set2')
plt.title('NZ Min Tuition Fee Distribution by Year')
plt.ylabel('Min Fee (NZD)')
plt.tight_layout()
plt.show()

## 6. SAC Rate Distribution and Spread

Tukey box plots and correlation across categories.

In [ ]:
# Tukey box plot — SAC rate distribution by category
cat_order = (sac_clean.groupby('category_code')['sac_rate_level2_excl_gst']
             .mean().sort_values(ascending=False).index.tolist())

plt.figure(figsize=(12, 6))
sns.boxplot(data=sac_clean, x='sac_rate_level2_excl_gst', y='category_code',
            order=cat_order, whis=1.5, color='skyblue')
sns.stripplot(data=sac_clean, x='sac_rate_level2_excl_gst', y='category_code',
              order=cat_order, color='black', alpha=0.55, size=5)
plt.title('Tukey Box Plot — NZ TEC SAC Rates by Category')
plt.xlabel('SAC Rate Level-2 (NZD, excl. GST)')
plt.ylabel('Category Code')
plt.tight_layout()
plt.show()

# Raw vs log distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sac_clean['sac_rate_level2_excl_gst'].dropna(), bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Raw SAC rate distribution')
axes[0].set_xlabel('NZD per EFTS')

axes[1].hist(np.log(sac_clean['sac_rate_level2_excl_gst'].dropna()), bins=20, color='darkorange', edgecolor='white')
axes[1].set_title('Log SAC rate distribution')
axes[1].set_xlabel('log(NZD per EFTS)')
plt.tight_layout()
plt.show()

## 7. Additional Statistical Checks

Growth rates, nonlinearity, and heterogeneity across cost categories.

In [ ]:
try:
    from scipy import stats
except ImportError:
    stats = None

# CAGR per category
sac_growth = (sac_pivot[[sac_pivot.columns[0], sac_pivot.columns[-1]]]
              .dropna()
              .copy())
sac_growth.columns = ['start', 'end']
n_years = sac_pivot.columns[-1] - sac_pivot.columns[0]
sac_growth['CAGR_%'] = ((sac_growth['end'] / sac_growth['start']) ** (1 / n_years) - 1) * 100
sac_growth['Abs Change'] = sac_growth['end'] - sac_growth['start']
print('SAC Rate CAGR by category:')
print(sac_growth.sort_values('CAGR_%', ascending=False).to_string())

# Levene / ANOVA across categories
groups = [g['sac_rate_level2_excl_gst'].dropna().values
          for _, g in sac_clean.groupby('category_code') if len(g) > 1]
if stats and len(groups) >= 2:
    lev_stat, lev_p = stats.levene(*groups, center='median')
    f_stat, f_p     = stats.f_oneway(*groups)
    print(f"\nLevene's test: F={lev_stat:.3f}, p={lev_p:.4g}")
    print(f'One-way ANOVA: F={f_stat:.3f}, p={f_p:.4g}')

# Fees: year-over-year growth
fees_yoy = fees_summary[['year', 'avg_of_min_fees']].copy().sort_values('year')
fees_yoy['YoY_%'] = fees_yoy['avg_of_min_fees'].pct_change() * 100
print('\nFees year-over-year growth:')
print(fees_yoy.to_string(index=False))

## Data Characteristics & First-Order Effects

**TEC SAC rates:** The NZ government subsidy (SAC rate) functions analogously to the AUS Commonwealth Contribution — it is the per-student subsidy paid to the institution at Level 2 (bachelor degree). Categories A (arts/business) carry the lowest subsidy; categories H (agriculture/environment), N (priority engineering), and V (science) carry the highest. Unlike AUS where JRG executed a one-year step change in 2021, NZ SAC rates increase gradually through annual TEC review cycles.

**Fee cap policy:** NZ has a government-imposed annual tuition fee cap (historically ~3 % p.a. increase allowed), which constrains fee volatility. The 2020 domestic fee cap was $12,000/year under the Fees Free policy (first year). This limits fee responsiveness to cost-band changes — unlike AUS where uncapped domestic fees in non-priority disciplines rose sharply post-JRG.

**Implication for DiD:** No discrete NZ-specific funding shock occurred in 2021. SAC rates show smooth growth consistent with inflation indexing. This supports NZ's validity as a control group.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
raw_nz_dir = next((r / 'data' / 'raw' / 'NZ Funding' for r in candidate_roots
                   if (r / 'data' / 'raw' / 'NZ Funding').exists()), None)
assert raw_nz_dir, 'Cannot find NZ Funding directory'

sac   = pd.read_csv(raw_nz_dir / 'NZ_TEC_SAC_rates_all_years.csv')
fees_s = pd.read_csv(raw_nz_dir / 'NZ_UniversitiesNZ_domestic_fees_summary.csv')

print('=== NZ Funding — Variable Summary ===')
print(f'SAC: {sac.shape[0]} rows | years {sac["year"].min()}–{sac["year"].max()} | {sac["category_code"].nunique()} categories')
print(f'Fee summary: {fees_s.shape[0]} rows | years {fees_s["year"].tolist()}')

# SAC skewness
sk = stats.skew(sac['sac_rate_level2_excl_gst'].dropna())
sk_log = stats.skew(np.log(sac['sac_rate_level2_excl_gst'].dropna()))
print(f'SAC rate skewness: raw={sk:.3f}  |  log={sk_log:.3f}')

# Trend: aggregate mean SAC over time
agg_sac = sac.groupby('year')['sac_rate_level2_excl_gst'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('NZ Funding — Data Characteristics & First-Order Effects', fontsize=12, fontweight='bold')

axes[0].plot(agg_sac['year'], agg_sac['sac_rate_level2_excl_gst'], marker='o', linewidth=2, color='steelblue')
axes[0].plot(fees_s['year'], fees_s['avg_of_min_fees'], marker='s', ls='--', linewidth=2, color='darkorange')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('NZD')
axes[0].set_title('A. Avg SAC Rate (blue) vs Avg Min Fee (orange)')
axes[0].legend(['Mean SAC rate', 'Mean min tuition fee'], fontsize=9)

axes[1].hist(np.log(sac['sac_rate_level2_excl_gst'].dropna()), bins=20, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log(SAC rate)')
axes[1].set_title(f'B. Log SAC rate distribution (skewness={sk_log:.2f})')

plt.tight_layout()
plt.show()

### What Is Learned

1. **Variable characteristics:** TODO — complete after running analysis.
2. **TEC SAC rate structure:** TODO — note the dispersion across cost categories and which fields receive high vs low subsidies.
3. **Fee cap context:** TODO — compare NZ fee growth rate to AUS post-JRG fee changes.
4. **Funding data gaps:** Fee data missing 2018–2021 (PDFs not publicly available); SAC rates complete 2016–2025.
5. **Implications for NZ as control group:** TODO — confirm absence of discrete NZ funding shock in 2021.